# Power Analysis — ARLMP Benchmark

A priori power analysis. Commit this notebook **before** any API calls.

**Study design:** Paired (each URL evaluated under every format × model combination)  
**Primary test:** Friedman test for overall format effect; Wilcoxon signed-rank for pairwise comparisons  
**Correction:** Holm-Bonferroni for 21 pairwise comparisons (C(7,2))  
**Target:** Detect ≥10% relative difference between adjacent compact formats  
**Alpha (family-wise):** 0.05 after correction  
**Target power:** ≥0.90

In [ ]:
# Install dependencies if needed
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'scipy', 'matplotlib', 'numpy', '-q'])

import numpy as np
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Libraries loaded.')

## 1. Parameters

In [ ]:
FORMAT_NAMES = ['JSON', 'minJSON', 'YAML', 'TOML', 'TOON', 'Markdown', 'ARLMP-Min']
MEAN_TOKENS  = [832,   701,      655,    630,    588,    512,        398]
STD_TOKENS   = [120,   100,       95,     90,     85,     75,         60]

N_FORMATS        = 7
N_COMPARISONS    = N_FORMATS * (N_FORMATS - 1) // 2  # 21
ALPHA_FAMILYWISE = 0.05
ALPHA_PER_TEST   = ALPHA_FAMILYWISE / N_COMPARISONS  # Bonferroni bound
TARGET_POWER     = 0.90
N_SIMULATIONS    = 5000  # 5k sims sufficient; 10k causes timeout in some kernels

idx_a = FORMAT_NAMES.index('TOON')
idx_b = FORMAT_NAMES.index('Markdown')

print(f'Hardest adjacent pair : TOON ({MEAN_TOKENS[idx_a]}) vs Markdown ({MEAN_TOKENS[idx_b]})')
print(f'Absolute difference   : {MEAN_TOKENS[idx_a]-MEAN_TOKENS[idx_b]} tokens')
print(f'Relative difference   : {(MEAN_TOKENS[idx_a]-MEAN_TOKENS[idx_b])/MEAN_TOKENS[idx_a]*100:.1f}%')
print(f'Alpha per test        : {ALPHA_PER_TEST:.5f} (Bonferroni bound, {N_COMPARISONS} comparisons)')

## 2. Token Count Power (Wilcoxon signed-rank, paired)

In [ ]:
def simulate_wilcoxon_power(mean_a, std_a, mean_b, std_b, n, alpha, n_sim=N_SIMULATIONS):
    rng = np.random.default_rng(42)
    rejections = 0
    for _ in range(n_sim):
        a = rng.normal(mean_a, std_a, n)
        b = rng.normal(mean_b, std_b, n)
        _, p = stats.wilcoxon(a, b, alternative='two-sided')
        if p < alpha:
            rejections += 1
    return rejections / n_sim

sample_sizes = [200, 300, 400, 500, 750, 1000, 1250, 1500]
powers = []

print(f'Simulating power for hardest pair: TOON vs Markdown')
print(f'n        power   pass?')
print('-' * 25)

min_n = None
for n in sample_sizes:
    power = simulate_wilcoxon_power(
        MEAN_TOKENS[idx_a], STD_TOKENS[idx_a],
        MEAN_TOKENS[idx_b], STD_TOKENS[idx_b],
        n=n, alpha=ALPHA_PER_TEST
    )
    powers.append(power)
    ok = '✓' if power >= TARGET_POWER else '✗'
    print(f'n={n:5d}   {power:.3f}   {ok}')
    if power >= TARGET_POWER and min_n is None:
        min_n = n

## 3. Accuracy Power (McNemar)

In [ ]:
def simulate_mcnemar_power(acc_a, acc_b, n, alpha, n_sim=N_SIMULATIONS):
    rng = np.random.default_rng(42)
    rejections = 0
    p_ab  = max(acc_a - acc_b, 0.001)
    p_ba  = max(acc_b - acc_a, 0.001)
    p_both = acc_a * acc_b
    p_neither = max(1 - p_both - p_ab - p_ba, 0.001)
    probs = np.array([p_both, p_ab, p_ba, p_neither])
    probs /= probs.sum()
    for _ in range(n_sim):
        counts = rng.multinomial(n, probs)
        b, c = counts[1], counts[2]
        if b + c == 0:
            continue
        chi2 = (abs(b - c) - 1) ** 2 / (b + c)
        p = 1 - stats.chi2.cdf(chi2, df=1)
        if p < alpha:
            rejections += 1
    return rejections / n_sim

print('Safety triage accuracy: JSON(0.942) vs ARLMP-Min(0.902)')
for n in [500, 750, 1000, 1250]:
    power = simulate_mcnemar_power(0.942, 0.902, n=n, alpha=ALPHA_PER_TEST)
    ok = '✓' if power >= TARGET_POWER else '✗'
    print(f'  n={n:5d}  power={power:.3f}  {ok}')

## 4. Power Curve Plot

In [ ]:
import os
os.makedirs('../codebook', exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sample_sizes, powers, 'ko-', linewidth=1.5, markersize=6)
ax.axhline(TARGET_POWER, color='gray', linestyle='--', linewidth=1,
           label=f'Target power = {TARGET_POWER}')
ax.axvline(1000, color='black', linestyle=':', linewidth=1,
           label='Planned n = 1,000')
ax.set_xlabel('Sample size (n links)')
ax.set_ylabel('Estimated power')
ax.set_title('Power curve — hardest pairwise comparison\n(TOON vs Markdown, token count)')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../codebook/power_curve.png', dpi=150)
plt.show()
print('Saved to codebook/power_curve.png')

## 5. Conclusion

In [ ]:
power_at_1000 = powers[sample_sizes.index(1000)]

print('=' * 55)
print('POWER ANALYSIS CONCLUSION')
print('=' * 55)
print(f'Hardest comparison : TOON vs Markdown (token count)')
print(f'Effect (absolute)  : {MEAN_TOKENS[idx_a]-MEAN_TOKENS[idx_b]} tokens')
print(f'Effect (relative)  : {(MEAN_TOKENS[idx_a]-MEAN_TOKENS[idx_b])/MEAN_TOKENS[idx_a]*100:.1f}%')
print(f'Alpha per test     : {ALPHA_PER_TEST:.5f} (Bonferroni bound, {N_COMPARISONS} comparisons)')
print(f'Target power       : {TARGET_POWER}')
print()
print(f'Minimum n for target power : {min_n}')
print(f'Power at planned n=1,000   : {power_at_1000:.3f}')
print()
if power_at_1000 >= TARGET_POWER:
    print(f'✓ n=1,000 is well-justified (power={power_at_1000:.3f} >> target {TARGET_POWER})')
    print(f'  (+200 buffer → sample 1,200 URLs to account for exclusions)')
else:
    print(f'✗ Increase n — power={power_at_1000:.3f} < target {TARGET_POWER}')
print('=' * 55)